In [ ]:
from ml_homework.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR


# Задача класифікації з логістичною регресією в ScikitLearn

В цьому ноутбуці ми
- розвʼяжемо задачу бінарної класифікації
- побудуємо ПОВНЕ і ПОВНОЦІННЕ рішення задачі машинного навчання з усіма основними етапами
- натренуємо логістичну регресію і виміряємо її точніть
- збережемо результати моделі, а також дані
- дізнаємось про пайплайни і навчимось ними користатись

## Постановка задачі

В цьому уроці ми навчимося застосовувати _логістичну регресію_ до реального набору даних з [Kaggle](https://kaggle.com/datasets):

**НАША ЗАДАЧА**: [Набір даних про дощі в Австралії](https://kaggle.com/jsphyg/weather-dataset-rattle-package) містить приблизно 10 років щоденних спостережень за погодою з численних австралійських метеостанцій. Ось невеликий зразок з набору даних:

![](https://i.imgur.com/5QNJvir.png)

Як Data Scientist у Бюро метеорології, вам доручено створити повністю автоматизовану систему, яка може використовувати дані про погоду на сьогодні для певного місця, щоб передбачити, чи буде дощ у цьому місці завтра.

![](https://i.imgur.com/KWfcpcO.png)


### Робочий процес машинного навчання

Незалежно від того, чи вирішуємо ми задачу регресії за допомогою лінійної регресії, чи задачу класифікації за допомогою логістичної регресії, робочий процес для навчання моделі є однаковим:

1. Ми ініціалізуємо модель з випадковими параметрами (вагами та зміщеннями).
2. Ми передаємо деякі вхідні дані в модель, щоб отримати прогнози.
3. Ми порівнюємо прогнози моделі з фактичними цільовими значеннями за допомогою функції втрат.
4. Ми використовуємо метод оптимізації (такий як метод найменших квадратів, градієнтний спуск тощо) для зменшення втрат шляхом коригування ваг та зміщень моделі.
5. Ми повторюємо кроки з 1 по 4, поки прогнози моделі не стануть достатньо точними.


<img src="https://i.imgur.com/g32CoIy.png" width="480">


## Завантаження даних

Ми будемо використовувати бібліотеку [`opendatasets`](https://github.com/JovianML/opendatasets) для завантаження даних з Kaggle безпосередньо в Jupyter. Давайте встановимо та імпортуємо `opendatasets`.

In [1]:
!pip install opendatasets --upgrade --quiet

In [2]:
import opendatasets as od

In [3]:
od.version()

'0.1.22'

Тепер набір даних можна завантажити за допомогою `od.download`. Коли ви виконаєте `od.download`, вас попросять вказати ваше ім'я користувача Kaggle та API ключ. Дотримуйтесь цих інструкцій, щоб створити API ключ: http://bit.ly/kaggle-creds

In [4]:
dataset_url = 'https://www.kaggle.com/jsphyg/weather-dataset-rattle-package'

In [ ]:
od.download(dataset_url)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds


Після виконання вищезазначеної команди, набір даних буде завантажено та розпаковано в директорію `weather-dataset-rattle-package`.

In [ ]:
import os

In [ ]:
data_dir = RAW_DATA_DIR / "weather"

In [ ]:
os.listdir(data_dir)

In [ ]:
train_csv = data_dir / "weatherAUS.csv"

Давайте завантажимо дані з `weatherAUS.csv` за допомогою Pandas.

In [ ]:
import pandas as pd

In [ ]:
# оберіть опцію, яка вам зручніша
# raw_df = pd.read_csv(train_csv)
raw_df = pd.read_csv(RAW_DATA_DIR / "weather/weatherAUS.csv")

In [ ]:
raw_df

Набір даних містить понад 145,000 рядків і 23 стовпці. Набір даних містить стовпці з датами, числовими та категоріальними значеннями. Наша мета - створити модель для прогнозування значення в стовпці `RainTomorrow`.

Давайте перевіримо типи даних та відсутні значення в різних стовпцях.

In [ ]:
raw_df.info()

Хоча ми повинні мати можливість заповнити відсутні значення для більшості стовпців, можливо, варто відкинути рядки, де значення `RainTomorrow` або `RainToday` відсутні, щоб зробити наш аналіз і моделювання простішими (оскільки одне з них є цільовою змінною, а інше, ймовірно, дуже тісно пов'язане з цільовою змінною).

In [ ]:
raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

In [ ]:
raw_df.shape

Як би ви впоралися з відсутніми значеннями в інших стовпцях?

## (Опціонально) Робота з вибіркою

Коли ви працюєте з великими наборами даних, що містять мільйони рядків, спочатку варто працювати з вибіркою, щоб швидко налаштувати свій блокнот для навчання моделі. Якщо ви хочете працювати з вибіркою, просто встановіть значення `use_sample` на `True`.

In [ ]:
use_sample = False

In [ ]:
sample_fraction = 0.1

In [ ]:
if use_sample:
    raw_df = raw_df.sample(frac=sample_fraction).copy()

Переконайтеся, що `use_sample` встановлено на `False`, і повторно запустіть ноутбук від початку до кінця, коли будете готові використовувати весь набір даних.

## Навчальна, валідаційна та тестова вибірки

В попередньому уроці ми дізнались, що в машинному навчанні варто розділяти вибірку на підвибірки: train, validation, test.

1. **Навчальна вибірка** - використовується для навчання моделі, тобто для обчислення втрат і налаштування ваг моделі за допомогою оптимізаційної техніки.


2. **Валідаційна вибірка** - використовується для оцінки моделі під час навчання, налаштування гіперпараметрів моделі (оптимізаційна техніка, регуляризація тощо) та вибору найкращої версії моделі. Вибір хорошої валідаційної вибірки є важливим для навчання моделей, які добре узагальнюють. [Дізнайтеся більше тут.](https://www.fast.ai/2017/11/13/validation-sets/)


3. **Тестова вибірка** - використовується для порівняння різних моделей або підходів і звітування про кінцеву точність моделі. Для багатьох наборів даних тестові вибірки надаються окремо. Тестова вибірка повинна максимально точно відображати тип даних, з якими модель зіткнеться в реальному світі.


<img src="https://i.imgur.com/j8eITrK.png" width="480">


Як загальне правило, рекомендується використовувати близько 60% даних для навчальної вибірки, 20% для валідаційної вибірки та 20% для тестової вибірки. Якщо окрема тестова вибірка вже надана, ви можете використовувати розподіл 75%-25% для навчання та валідації.

Давайте зробимо розбиття нашої вибірки.

Коли рядки в наборі даних не мають внутрішнього порядку, звичайною практикою є вибір **випадкових** підмножин рядків для створення тестових і валідаційних вибірок.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_val_df, test_df = train_test_split(raw_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

In [ ]:
print('train_df.shape :', train_df.shape)
print('val_df.shape :', val_df.shape)
print('test_df.shape :', test_df.shape)

Однак, під час роботи з датами часто краще розділяти навчальні, валідаційні та тестові набори за часом, щоб модель навчалася на даних з минулого і оцінювалася на даних з майбутнього.

Для поточного набору даних ми можемо використати стовпець Date у наборі даних, щоб створити інший стовпець для року. Ми виберемо останні два роки для тестового набору і один рік перед ними для валідаційного набору.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.title('No. of Rows per Year')
sns.countplot(x=pd.to_datetime(raw_df.Date).dt.year);

In [ ]:
year = pd.to_datetime(raw_df.Date).dt.year

train_df = raw_df[year < 2015]
val_df = raw_df[year == 2015]
test_df = raw_df[year > 2015]

In [ ]:
print('train_df.shape :', train_df.shape)
print('val_df.shape :', val_df.shape)
print('test_df.shape :', test_df.shape)

Хоча це не ідеальний розподіл 60-20-20, ми забезпечили, щоб набори даних для тестування та валідації містили дані за всі 12 місяців року.

## Визначення вхідних та цільових стовпців

Часто не всі стовпці в наборі даних корисні для навчання моделі. У поточному наборі даних ми можемо ігнорувати стовпець `Date`, оскільки ми хочемо використовувати лише погодні умови для прогнозування, чи буде дощ наступного дня. В загальному випадку дату ми можемо використовувати і генерувати з неї багато ознак, наприклад, місяць, сезон, чи є день буднім і тд.

Давайте створимо список вхідних стовпців, а також визначимо цільовий стовпець.

In [ ]:
input_cols = list(train_df.columns)[1:-1]
target_col = 'RainTomorrow'

In [ ]:
print(input_cols)

In [ ]:
target_col

Тепер ми можемо створити вхідні дані та цільові значення для навчальних, валідаційних та тестових наборів для подальшої обробки та навчання моделі.

In [ ]:
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()

In [ ]:
val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_col].copy()

In [ ]:
test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

Давайте також визначимо, які з колонок є числовими, а які категоріальними. Це буде корисно пізніше, оскільки нам потрібно буде перетворити категоріальні дані на числа для навчання моделі логістичної регресії.

In [ ]:
import numpy as np

In [ ]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

In [ ]:
numeric_cols, categorical_cols

Давайте переглянемо деяку статистику для числових стовпців.

In [ ]:
train_inputs[numeric_cols].describe().round(2)

Чи здаються діапазони числових стовпців розумними? Якщо ні, можливо, нам доведеться також провести очищення даних.

Давайте також перевіримо кількість категорій у кожному з категоріальних стовпців.

In [ ]:
train_inputs[categorical_cols].nunique()

## Заповнення відсутніх числових даних

Багато моделей машинного навчання не можуть працювати з відсутніми числовими даними. Процес заповнення відсутніх значень називається імпутацією.

<img src="https://i.imgur.com/W7cfyOp.png" width="480">

Існує кілька технік імпутації, але ми використаємо найпростішу: заміну відсутніх значень на середнє значення в стовпці за допомогою класу `SimpleImputer` з `sklearn.impute`.

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
?SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy = 'mean')

Перш ніж виконувати імпутацію, давайте перевіримо кількість відсутніх значень у кожному числовому стовпці.

In [ ]:
raw_df.shape

In [ ]:
raw_df[numeric_cols].isna().sum()

Ці значення розподілені між навчальним, тестовим та валідаційним наборами. Ви також можете перевірити кількість відсутніх значень окремо для `train_inputs`, `val_inputs` та `test_inputs`.

In [ ]:
train_inputs[numeric_cols].isna().sum()

Першим кроком в імпутації є `fit` імпутера до даних, тобто обчислення обраної статистики (наприклад, середнього) для кожного стовпця в наборі даних.

In [ ]:
imputer.fit(train_inputs[numeric_cols])

Після виклику `fit`, обчислена статистика для кожного стовпця зберігається у властивості `statistics_` об'єкта `imputer`.

In [ ]:
list(imputer.statistics_)

Тепер відсутні значення в наборах даних для тренування, тестування та валідації можна заповнити за допомогою методу `transform` об'єкта `imputer`.

In [ ]:
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

Відсутні значення тепер заповнені середнім значенням кожного стовпця.

In [ ]:
train_inputs[numeric_cols].isna().sum()

Ви можете дізнатися більше про інші методи імпутації тут: https://scikit-learn.org/stable/modules/impute.html

Метод імпутації - це частина експерименту, адже результат моделі може змінитись від способа імпутації даних. Тому ми змінюючи імпутацію можемо спостерігати, як це впливає на модель.

## Масштабування числових ознак

Ми памʼятаємо з вами про масштабування ознак з минулої лекції. Масштабування числових ознак забезпечує, що жодна конкретна ознака не має непропорційного впливу на втрати моделі. Оптимізаційні алгоритми також працюють краще на практиці з меншими числами.

Числові стовпці в нашому наборі даних мають різні діапазони.

In [ ]:
raw_df[numeric_cols].describe()

Давайте використаємо `MinMaxScaler` з `sklearn.preprocessing`, щоб масштабувати значення до діапазону $(0,1)$.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
?MinMaxScaler

In [ ]:
scaler = MinMaxScaler()

Спочатку ми `підганяємо` масштабувальник до даних, тобто обчислюємо діапазон значень для кожного числового стовпця.

In [ ]:
scaler.fit(train_inputs[numeric_cols])

Тепер ми можемо окремо масштабувати навчальні, валідаційні та тестові набори, використовуючи метод `transform` класу `scaler`.

In [ ]:
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

Тепер ми можемо перевірити, що значення в кожному стовпці лежать у діапазоні $(0,1)$

In [ ]:
train_inputs[numeric_cols].describe().round(2)

## Кодування категоріальних даних

Оскільки моделі машинного навчання можуть бути навчені лише з числовими даними, нам потрібно перетворити категоріальні дані на числа. Поширеною технікою є використання one-hot кодування для категоріальних стовпців.

<img src="https://i.imgur.com/n8GuiOO.png" width="640">

One-hot кодування передбачає додавання нового бінарного (0/1) стовпця для кожної унікальної категорії категоріального стовпця.

In [ ]:
raw_df[categorical_cols].nunique()

Ми можемо виконати one hot кодування за допомогою класу `OneHotEncoder` з `sklearn.preprocessing`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
?OneHotEncoder

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

Спочатку ми `fit` енкодер до даних, тобто визначаємо повний список категорій у всіх категоріальних стовпцях.

In [ ]:
encoder.fit(train_inputs[categorical_cols])

In [ ]:
encoder.categories_

Енкодер створив список категорій для кожного з категоріальних стовпців у наборі даних.

Ми можемо згенерувати назви стовпців для кожної окремої категорії, використовуючи `get_feature_names_out`.

In [ ]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
print(encoded_cols)

Усі вищезазначені стовпці будуть додані до `train_inputs`, `val_inputs` та `test_inputs`.

Для виконання кодування ми використовуємо метод `transform` об'єкта `encoder`.

In [ ]:
encoder.transform(train_inputs[categorical_cols])[:10]

In [ ]:
train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

Ми можемо перевірити, що ці нові стовпці були додані до наших навчальних, тестових та валідаційних наборів.

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
test_inputs

## Збереження оброблених даних на диск

Може бути корисним зберігати оброблені дані на диск, особливо для дуже великих наборів даних, щоб уникнути повторення кроків попередньої обробки щоразу, коли ви запускаєте Jupyter notebook. Формат parquet є швидким та ефективним форматом для збереження та завантаження dataframes Pandas. Дані зберігаються і читаються швидше, ніж з csv, а також менше важать!

Більше про паркет і формати збереження даних [тут](https://towardsdatascience.com/demystifying-the-parquet-file-format-13adb0206705)

In [ ]:
print('train_inputs:', train_inputs.shape)
print('train_targets:', train_targets.shape)
print('val_inputs:', val_inputs.shape)
print('val_targets:', val_targets.shape)
print('test_inputs:', test_inputs.shape)
print('test_targets:', test_targets.shape)

In [ ]:
!pip install pyarrow --quiet

In [ ]:
train_inputs.to_parquet(PROCESSED_DATA_DIR / "train_inputs.parquet")
val_inputs.to_parquet(PROCESSED_DATA_DIR / "val_inputs.parquet")
test_inputs.to_parquet(PROCESSED_DATA_DIR / "test_inputs.parquet")

In [ ]:
%%time
pd.DataFrame(train_targets).to_parquet(PROCESSED_DATA_DIR / "train_targets.parquet")
pd.DataFrame(val_targets).to_parquet(PROCESSED_DATA_DIR / "val_targets.parquet")
pd.DataFrame(test_targets).to_parquet(PROCESSED_DATA_DIR / "test_targets.parquet")

Ми можемо зчитати дані назад, використовуючи `pd.read_parquet`.

In [ ]:
%%time

train_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "train_inputs.parquet")
val_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "val_inputs.parquet")
test_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "test_inputs.parquet")

train_targets = pd.read_parquet(PROCESSED_DATA_DIR / "train_targets.parquet")[target_col]
val_targets = pd.read_parquet(PROCESSED_DATA_DIR / "val_targets.parquet")[target_col]
test_targets = pd.read_parquet(PROCESSED_DATA_DIR / "test_targets.parquet")[target_col]

Давайте перевіримо, чи дані були завантажені правильно.

In [ ]:
print('train_inputs:', train_inputs.shape)
print('train_targets:', train_targets.shape)
print('val_inputs:', val_inputs.shape)
print('val_targets:', val_targets.shape)
print('test_inputs:', test_inputs.shape)
print('test_targets:', test_targets.shape)

In [ ]:
val_inputs

In [ ]:
val_targets

## Навчання моделі логістичної регресії

Логістична регресія є загальновживаною технікою для розв'язання задач бінарної класифікації. У моделі логістичної регресії:

- ми беремо лінійну комбінацію (або зважену суму вхідних ознак)
- ми застосовуємо сигмоїдну функцію до результату, щоб отримати число між 0 і 1
- це число представляє ймовірність того, що вхідні дані будуть класифіковані як "Так"
- замість RMSE використовується функція втрат крос-ентропії для оцінки результатів

Ось візуальне резюме того, як структурована модель логістичної регресії:

<img src="https://i.imgur.com/YMaMo5D.png" width="480">

Сигмоїдна функція, застосована до лінійної комбінації вхідних даних, має наступну формулу:

<img src="https://i.imgur.com/sAVwvZP.png" width="400">

Для навчання моделі логістичної регресії ми можемо використовувати клас `LogisticRegression` з бібліотеки Scikit-learn.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(solver='liblinear')

Ми можемо тренувати модель за допомогою `model.fit`.

In [ ]:
?model.fit

In [ ]:
model.fit(train_inputs[numeric_cols + encoded_cols], train_targets)

`model.fit` використовує наступний робочий процес для навчання моделі ([джерело](https://www.deepnetts.com/blog/from-basic-machine-learning-to-deep-learning-in-5-minutes.html)):

1. Ми ініціалізуємо модель з випадковими параметрами (вагами та зміщеннями).
2. Ми передаємо деякі вхідні дані в модель, щоб отримати прогнози.
3. Ми порівнюємо прогнози моделі з фактичними цілями за допомогою функції втрат.
4. Ми використовуємо метод оптимізації (наприклад, метод найменших квадратів, градієнтний спуск тощо), щоб зменшити втрати, коригуючи ваги та зміщення моделі.
5. Ми повторюємо кроки з 1 по 4, поки прогнози моделі не стануть достатньо точними.

<img src="https://i.imgur.com/g32CoIy.png" width="480">

Логістичну регресію також можна застосувати до багатокласових задач класифікації з деякими модифікаціями. Про це - далі.

Давайте перевіримо ваги та зміщення навченої моделі.

In [ ]:
print(numeric_cols + encoded_cols)

In [ ]:
model.coef_.round(2)

In [ ]:
print(model.coef_.tolist())

In [ ]:
print(model.intercept_)

Кожна вага застосовується до значення в конкретній колонці вхідних даних. Чим більша вага, тим більший вплив колонки на прогноз.

Ваги лог. регресії можна інтерпретувати!
Як - детально тут: https://youtu.be/vN5cNN2-HWE?si=CEJxeORcbTgjfReY

## Прогнозування та оцінка моделі

Тепер ми можемо використовувати навчальну модель для прогнозування на навчальній, тестовій

In [ ]:
X_train = train_inputs[numeric_cols + encoded_cols]
X_val = val_inputs[numeric_cols + encoded_cols]
X_test = test_inputs[numeric_cols + encoded_cols]

In [ ]:
X_train.head()

In [ ]:
train_preds = model.predict(X_train)

In [ ]:
train_preds

In [ ]:
train_targets

Ми можемо вивести ймовірнісний прогноз за допомогою `predict_proba`.

In [ ]:
train_probs = model.predict_proba(X_train)
train_probs

In [ ]:
np.argmax(train_probs, axis=1)

In [ ]:
plt.hist(train_probs[:,1])

In [ ]:
(train_probs[:,1][:20]>0.2).astype(int)

In [ ]:
(train_probs[:,1][:20]>0.5).astype(int)

In [ ]:
train_probs[:,1].round(2)[:20]
#  >0.2).astype(int)

In [ ]:
train_targets.value_counts(normalize=True)

Числа вище вказують на ймовірності для цільових класів "Ні" та "Так".

In [ ]:
model.classes_

Ми можемо перевірити точність прогнозів моделі, обчисливши відсоток співпадаючих значень у `train_preds` та `train_targets`.

Це можна зробити за допомогою функції `accuracy_score` з `sklearn.metrics`.

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(train_targets, train_preds)

Модель досягає точності 85,1% на навчальному наборі. Ми можемо візуалізувати розподіл правильно та неправильно класифікованих входів за допомогою матриці плутанини (confusion matrix).

<img src="https://i.imgur.com/UM28BCN.png" width="480">

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(train_targets, train_preds)

Давайте визначимо допоміжну функцію для генерації прогнозів, обчислення точності та побудови матриці невідповідностей для заданого набору вхідних даних.

In [ ]:
def predict_and_plot(inputs, targets, name=''):
    preds = model.predict(inputs)

    accuracy = accuracy_score(targets, preds)
    print("Accuracy: {:.2f}%".format(accuracy * 100))

    cf = confusion_matrix(targets, preds, normalize='true')
    plt.figure()
    sns.heatmap(cf, annot=True)
    plt.xlabel('Prediction')
    plt.ylabel('Target')
    plt.title('{} Confusion Matrix'.format(name));

    return preds

In [ ]:
train_preds = predict_and_plot(X_train, train_targets, 'Training')

Давайте обчислимо точність моделі на валідаційному та тестовому наборах даних також.

In [ ]:
val_preds = predict_and_plot(X_val, val_targets, 'Validation')

In [ ]:
test_preds = predict_and_plot(X_test, test_targets, 'Test')

Точність моделі на тестовому та валідаційному наборах перевищує 84%, що свідчить про те, що наша модель добре узагальнює дані, які вона раніше не бачила.

Але наскільки хороша точність 84%? Хоча це залежить від природи проблеми та бізнес-вимог, хороший спосіб перевірити, чи модель дійсно навчилася чомусь корисному, — це порівняти її результати з "випадковою" або "дурною" моделлю.

Давайте створимо дві моделі: одну, яка вгадує випадково, і іншу, яка завжди повертає "Ні". Обидві ці моделі повністю ігнорують вхідні дані, які їм надаються.

In [ ]:
def random_guess(inputs):
    return np.random.choice(["No", "Yes"], len(inputs))

In [ ]:
def all_no(inputs):
    return np.full(len(inputs), "No")

Давайте перевіримо точність цих двох моделей на тестовому наборі даних.

In [ ]:
accuracy_score(test_targets, random_guess(X_test))

In [ ]:
accuracy_score(test_targets, all_no(X_test))

Наша випадкова модель досягає точності 50%, а наша модель "завжди Ні" досягає точності 77%.

На щастя, наша модель краща за "дурну" або "випадкову" модель! Це не завжди так, тому гарною практикою є порівнювати будь-яку модель, яку ви тренуєте, з такими базовими моделями.

# Precision, Recall, F1-score
Ми з вами помітили, що використання Accuracy метрики не дає нам надто точну оцінку точності моделі, адже для тривіальної моделі, яка завжди каже "Ні" точність дорівнює аж 77%. Ми вже навчились користуватись confusion matrix, але яка ми вже обговорили раніше, в ML-експериментах основна метрика має бути одна. І в задачах бінарної класифікації цією метрикою часто є F1-score. Що ж це за метрика?

F1 метрика поєднує в собі точність (precision) та повноту (recall) для створення єдиного значення, яке балансує між цими двома показниками. Це особливо корисно, коли необхідно оцінити модель за умов незбалансованих даних.

## Визначення
- **Точність (Precision)**: Відсоток правильно передбачених позитивних прикладів серед усіх прикладів, передбачених як позитивні.
  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$
  де $$ TP $$ (True Positives) — кількість правильно передбачених позитивних випадків, $$ FP $$ (False Positives) — кількість неправильно передбачених позитивних випадків.

- **Повнота (Recall)**: Відсоток правильно передбачених позитивних прикладів серед усіх реальних позитивних прикладів.
  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$
  де $$ FN $$ (False Negatives) — кількість неправильно передбачених негативних випадків.

## Обчислення F1 метрики
F1 метрика розраховується як гармонічне середнє між точністю та повнотою:
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

## Приклад розрахунку
Розглянемо простий приклад

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*ua3zIJ-koDQK6fRageGgFw.png)


In [ ]:
0.66*0.22/(0.66+0.22)



Отже, F1 метрика для даної моделі становить приблизно 0.165. Це значення вказує на загальну ефективність моделі в умовах, коли ми однаково враховуємо як помилки першого, так і другого роду (false positives та false negatives).

![](https://en.wikipedia.org/wiki/Precision_and_recall)



Більше прикладів і детальніше розібратись можна з цією [статтею](https://towardsdatascience.com/should-i-look-at-precision-recall-or-specificity-sensitivity-3946158aace1).

## Обчислення F1 для нашої моделі




In [ ]:
from sklearn.metrics import f1_score

In [ ]:
preds = model.predict(X_train)
preds[:5], train_targets[:5]

In [ ]:
f1_score(train_targets, preds, pos_label='Yes')

In [ ]:
def get_f1_score(inputs, targets, name=''):
  preds = model.predict(inputs)

  f1_score_ = f1_score(targets, preds, pos_label='Yes')
  print(f"F1 score {name}: {f1_score_:.2f}%")

get_f1_score(X_train, train_targets, 'Training')
get_f1_score(X_val, val_targets, 'Validation')
get_f1_score(X_test, test_targets, 'Test')

## Побудова ROC Curve та обчислення AUROC для нашої моделі


In [ ]:
from sklearn.metrics import roc_curve, auc

def compute_auroc_and_build_roc(inputs, targets, name=''):
  # Predict probabilities
  y_pred_proba = model.predict_proba(inputs)[:, 1]

  # Compute ROC curve
  fpr, tpr, thresholds = roc_curve(targets, y_pred_proba, pos_label='Yes')

    # Compute AUROC
  roc_auc = auc(fpr, tpr)
  print(f'AUROC for {name}: {roc_auc:.2f}')

  # Plot the ROC curve
  plt.figure()
  plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
  plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.title(f'Receiver Operating Characteristic (ROC) Curve for {name}')
  plt.legend(loc="lower right")
  plt.show()

In [ ]:
compute_auroc_and_build_roc(X_train, train_targets, 'Training')
compute_auroc_and_build_roc(X_val, val_targets, 'Validation')

## Прогнозування на основі одного введення

Після того, як модель була навчена до задовільної точності, її можна використовувати для прогнозування нових даних. Розглянемо наступний словник, що містить дані, зібрані сьогодні від метеорологічного департаменту Кетрін.

In [ ]:
new_input = {'Date': '2021-06-19',
             'Location': 'Katherine',
             'MinTemp': 23.2,
             'MaxTemp': 33.2,
             'Rainfall': 10.2,
             'Evaporation': 4.2,
             'Sunshine': np.nan,
             'WindGustDir': 'NNW',
             'WindGustSpeed': 52.0,
             'WindDir9am': 'NW',
             'WindDir3pm': 'NNE',
             'WindSpeed9am': 13.0,
             'WindSpeed3pm': 20.0,
             'Humidity9am': 89.0,
             'Humidity3pm': 58.0,
             'Pressure9am': 1004.8,
             'Pressure3pm': 1001.5,
             'Cloud9am': 8.0,
             'Cloud3pm': 5.0,
             'Temp9am': 25.7,
             'Temp3pm': 33.0,
             'RainToday': 'Yes'}

Перший крок - перетворити словник у Pandas dataframe, подібний до `raw_df`. Це можна зробити, передавши список, що містить даний словник, до конструктора `pd.DataFrame`.

In [ ]:
new_input_df = pd.DataFrame([new_input])

In [ ]:
new_input_df

Ми щойно створили Pandas dataframe з тими ж стовпцями, що й `raw_df` (за винятком `RainTomorrow`, який потрібно передбачити). Dataframe містить лише один рядок даних, що містить заданий вхід.

Тепер ми повинні застосувати ті самі перетворення, які застосовувалися під час навчання моделі:

1. Імпутація відсутніх значень за допомогою раніше створеного `imputer`
2. Масштабування числових ознак за допомогою раніше створеного `scaler`
3. Кодування категоріальних ознак за допомогою раніше створеного `encoder`

In [ ]:
new_input_df[numeric_cols] = imputer.transform(new_input_df[numeric_cols])
new_input_df[numeric_cols] = scaler.transform(new_input_df[numeric_cols])
new_input_df[encoded_cols] = encoder.transform(new_input_df[categorical_cols])

In [ ]:
X_new_input = new_input_df[numeric_cols + encoded_cols]
X_new_input

Тепер ми можемо зробити прогноз за допомогою `model.predict`.

In [ ]:
prediction = model.predict(X_new_input)[0]

In [ ]:
prediction

Наша модель прогнозує, що завтра в Кетрін буде дощ! Ми також можемо перевірити ймовірність цього прогнозу.

In [ ]:
prob = model.predict_proba(X_new_input)[0]

In [ ]:
prob

Здається, наша модель не надто впевнена у своєму прогнозі!

Давайте визначимо допоміжну функцію для здійснення прогнозів для окремих вхідних даних.

In [ ]:
def predict_input(single_input):
    input_df = pd.DataFrame([single_input])
    input_df[numeric_cols] = imputer.transform(input_df[numeric_cols])
    input_df[numeric_cols] = scaler.transform(input_df[numeric_cols])
    input_df[encoded_cols] = encoder.transform(input_df[categorical_cols])
    X_input = input_df[numeric_cols + encoded_cols]
    pred = model.predict(X_input)[0]
    prob = model.predict_proba(X_input)[0][list(model.classes_).index(pred)]
    return pred, prob

Тепер ми можемо використовувати цю функцію для здійснення прогнозів для окремих вхідних даних.

In [ ]:
new_input = {'Date': '2021-06-19',
             'Location': 'Launceston',
             'MinTemp': 23.2,
             'MaxTemp': 33.2,
             'Rainfall': 10.2,
             'Evaporation': 4.2,
             'Sunshine': np.nan,
             'WindGustDir': 'NNW',
             'WindGustSpeed': 52.0,
             'WindDir9am': 'NW',
             'WindDir3pm': 'NNE',
             'WindSpeed9am': 13.0,
             'WindSpeed3pm': 20.0,
             'Humidity9am': 89.0,
             'Humidity3pm': 58.0,
             'Pressure9am': 1004.8,
             'Pressure3pm': 1001.5,
             'Cloud9am': 8.0,
             'Cloud3pm': 5.0,
             'Temp9am': 25.7,
             'Temp3pm': 33.0,
             'RainToday': 'Yes'}

In [ ]:
predict_input(new_input)

## Збереження та завантаження навчених моделей

Ми можемо зберегти параметри (ваги та зміщення) нашої навченої моделі на диск, щоб не потрібно було перенавчати модель з нуля кожного разу, коли ми хочемо її використовувати. Разом з моделлю також важливо зберегти імпутери, скейлери, енкодери та навіть назви колонок. Все, що буде необхідно для генерації прогнозів за допомогою моделі, повинно бути збережено.

Ми можемо використовувати модуль `joblib` для збереження та завантаження об'єктів Python на диск.

In [ ]:
import joblib

Спочатку створимо словник, що містить усі необхідні об'єкти.

In [ ]:
aussie_rain = {
    'model': model,
    'imputer': imputer,
    'scaler': scaler,
    'encoder': encoder,
    'input_cols': input_cols,
    'target_col': target_col,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'encoded_cols': encoded_cols
}

Тепер ми можемо зберегти це у файл, використовуючи `joblib.dump`

In [ ]:
joblib.dump(aussie_rain, PROCESSED_DATA_DIR / "aussie_rain.joblib")

Об'єкт можна завантажити назад за допомогою `joblib.load`

In [ ]:
aussie_rain2 = joblib.load(PROCESSED_DATA_DIR / "aussie_rain.joblib")

Давайте використаємо завантажену модель для прогнозування на оригінальному тестовому наборі.

In [ ]:
test_preds2 = aussie_rain2['model'].predict(X_test)
accuracy_score(test_targets, test_preds2)

Як і очікувалося, ми отримуємо той самий результат, що й оригінальна модель.

## Підсумок

Хоча ми розглянули багато матеріалу в цьому уроці, кількість рядків коду для обробки даних і навчання моделі є досить невеликою. Кожен крок вимагає не більше 3-4 рядків коду.

### Попередня обробка даних

In [ ]:
import opendatasets as od
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Download the dataset
# od.download('https://www.kaggle.com/jsphyg/weather-dataset-rattle-package')
# raw_df = pd.read_csv(RAW_DATA_DIR / "weather/weatherAUS.csv")
raw_df = pd.read_csv(RAW_DATA_DIR / "weather/weatherAUS.csv")
raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

# Create training, validation and test sets
year = pd.to_datetime(raw_df.Date).dt.year
train_df, val_df, test_df = raw_df[year < 2015], raw_df[year == 2015], raw_df[year > 2015]

# Create inputs and targets
input_cols = list(train_df.columns)[1:-1]
target_col = 'RainTomorrow'
train_inputs, train_targets = train_df[input_cols].copy(), train_df[target_col].copy()
val_inputs, val_targets = val_df[input_cols].copy(), val_df[target_col].copy()
test_inputs, test_targets = test_df[input_cols].copy(), test_df[target_col].copy()

# Identify numeric and categorical columns
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

# Impute missing numerical values
imputer = SimpleImputer(strategy = 'mean').fit(train_inputs[numeric_cols])
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

# Scale numeric features
scaler = MinMaxScaler().fit(train_inputs[numeric_cols])
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(train_inputs[categorical_cols])
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

# Save processed data to disk
train_inputs.to_parquet(PROCESSED_DATA_DIR / "train_inputs.parquet")
val_inputs.to_parquet(PROCESSED_DATA_DIR / "val_inputs.parquet")
test_inputs.to_parquet(PROCESSED_DATA_DIR / "test_inputs.parquet")
pd.DataFrame(train_targets).to_parquet(PROCESSED_DATA_DIR / "train_targets.parquet")
pd.DataFrame(val_targets).to_parquet(PROCESSED_DATA_DIR / "val_targets.parquet")
pd.DataFrame(test_targets).to_parquet(PROCESSED_DATA_DIR / "test_targets.parquet")

# Load processed data from disk
train_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "train_inputs.parquet")
val_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "val_inputs.parquet")
test_inputs = pd.read_parquet(PROCESSED_DATA_DIR / "test_inputs.parquet")
train_targets = pd.read_parquet(PROCESSED_DATA_DIR / "train_targets.parquet")[target_col]
val_targets = pd.read_parquet(PROCESSED_DATA_DIR / "val_targets.parquet")[target_col]
test_targets = pd.read_parquet(PROCESSED_DATA_DIR / "test_targets.parquet")[target_col]


> **ВПРАВА**: Спробуйте пояснити кожен рядок коду в наведеній вище комірці своїми словами. Прокрутіть назад до відповідних розділів зошита, якщо потрібно.

### Навчання та оцінка моделі

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import joblib

# Select the columns to be used for training/prediction
X_train = train_inputs[numeric_cols + encoded_cols]
X_val = val_inputs[numeric_cols + encoded_cols]
X_test = test_inputs[numeric_cols + encoded_cols]

# Create and train the model
model = LogisticRegression(solver='liblinear')
model.fit(X_train, train_targets)

# Helper function to predict, compute accuracy & plot confustion matrix
def predict_and_plot(inputs, targets, name=''):
    preds = model.predict(inputs)
    f1 = f1_score(targets, preds, pos_label='Yes')
    print("F1 score: {:.2f}%".format(f1 * 100))
    cf = confusion_matrix(targets, preds, normalize='true')
    plt.figure()
    sns.heatmap(cf, annot=True)
    plt.xlabel('Prediction')
    plt.ylabel('Target')
    plt.title('{} Confusion Matrix'.format(name));
    return preds

# Evaluate on validation and test set
val_preds = predict_and_plot(X_val, val_targets, 'Validation')
test_preds = predict_and_plot(X_test, test_targets, 'Test')

# Save the trained model & load it back
aussie_rain = {'model': model, 'imputer': imputer, 'scaler': scaler, 'encoder': encoder,
               'input_cols': input_cols, 'target_col': target_col, 'numeric_cols': numeric_cols,
               'categorical_cols': categorical_cols, 'encoded_cols': encoded_cols}
joblib.dump(aussie_rain, PROCESSED_DATA_DIR / "aussie_rain.joblib")
aussie_rain2 = joblib.load(PROCESSED_DATA_DIR / "aussie_rain.joblib")

> **ВПРАВА**: Спробуйте пояснити кожен рядок коду в наведеній вище комірці своїми словами. Прокрутіть назад до відповідних розділів зошита, якщо потрібно.

### Прогнозування на окремих вхідних даних

In [ ]:
def predict_input(single_input):
    input_df = pd.DataFrame([single_input])
    input_df[numeric_cols] = imputer.transform(input_df[numeric_cols])
    input_df[numeric_cols] = scaler.transform(input_df[numeric_cols])
    input_df[encoded_cols] = encoder.transform(input_df[categorical_cols])
    X_input = input_df[numeric_cols + encoded_cols]
    pred = model.predict(X_input)[0]
    prob = model.predict_proba(X_input)[0][list(model.classes_).index(pred)]
    return pred, prob

new_input = {'Date': '2021-06-19',
             'Location': 'Launceston',
             'MinTemp': 23.2,
             'MaxTemp': 33.2,
             'Rainfall': 10.2,
             'Evaporation': 4.2,
             'Sunshine': np.nan,
             'WindGustDir': 'NNW',
             'WindGustSpeed': 52.0,
             'WindDir9am': 'NW',
             'WindDir3pm': 'NNE',
             'WindSpeed9am': 13.0,
             'WindSpeed3pm': 20.0,
             'Humidity9am': 89.0,
             'Humidity3pm': 58.0,
             'Pressure9am': 1004.8,
             'Pressure3pm': 1001.5,
             'Cloud9am': 8.0,
             'Cloud3pm': 5.0,
             'Temp9am': 25.7,
             'Temp3pm': 33.0,
             'RainToday': 'Yes'}

predict_input(new_input)


# Пайплайни
В цій роботі ми для тренування моделі пройшли кілька кроків попередньої обробки даних: заповнення пропущених значень, масштабування числових ознак, кодування категоріальних ознак. Після кожного кроку в нас створюються нові структури даних і ми маємо їх записати. Також в якийсь момент ми можемо припуститись помилки і забути запустити якисьй крок.

Тому для багатоетапної попередньої обробки даних часто використовують паймплайни - `sklearn piplines`. І це дуже зручно, та значно зменшує кількість коду.

### Що таке пайплайни в `sklearn`?

Пайплайни в `sklearn` є потужним інструментом, який допомагає об'єднати кілька кроків обробки даних та навчання моделі в один цілісний об'єкт. Це дозволяє спростити та автоматизувати робочий процес машинного навчання, забезпечуючи:

1. **Управління послідовністю кроків**: Всі етапи обробки та навчання можуть бути визначені та виконані в певному порядку.
2. **Уніфікований інтерфейс**: Пайплайни надають єдиний інтерфейс для налаштування параметрів, навчання та оцінки моделі.
3. **Зменшення ризику витоку даних**: Пайплайни дозволяють правильно розділяти обробку даних та навчання моделі, зменшуючи ризик витоку даних між тренувальними та тестовими наборами.

### Простий приклад пайплайна

Розглянемо простий приклад створення пайплайна для задачі бінарної класифікації з використанням стандартного набору даних `breast_cancer` з `sklearn`. Цей набір даних містить різні характеристики клітин та інформацію про те, чи є вони злоякісними або доброякісними.


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Завантажуємо дані
data = load_breast_cancer()
X, y = data.data, data.target

# Розділяємо дані на тренувальний та тестовий набори
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Створюємо пайплайн
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Масштабування даних
    ('classifier', LogisticRegression())  # Логістична регресія
])

# Навчаємо модель
pipeline.fit(X_train, y_train)


In [ ]:
pipeline.named_steps['scaler'].transform(X_test)

In [ ]:
# Робимо прогнози
y_pred = pipeline.predict(X_test)

# Оцінюємо модель
f1 = f1_score(y_test, y_pred)
print(f'F1 Score: {f1:.2f}')

# Виводимо звіт про класифікацію
print(classification_report(y_test, y_pred, target_names=data.target_names))

**Пайплайни - це зручно!**

Нижче показую, як ми можемо нашу розробку зібрати у пайплайн.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Download the dataset
# od.download('https://www.kaggle.com/jsphyg/weather-dataset-rattle-package')
# raw_df = pd.read_csv(RAW_DATA_DIR / "weather/weatherAUS.csv")
raw_df = pd.read_csv(RAW_DATA_DIR / "weather/weatherAUS.csv")
raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

# Create training, validation and test sets
year = pd.to_datetime(raw_df.Date).dt.year
train_df, val_df, test_df = raw_df[year < 2015], raw_df[year == 2015], raw_df[year > 2015]

# Create inputs and targets
input_cols = list(train_df.columns)[1:-1]
target_col = 'RainTomorrow'
train_inputs, train_targets = train_df[input_cols], train_df[target_col]
val_inputs, val_targets = val_df[input_cols], val_df[target_col]
test_inputs, test_targets = test_df[input_cols], test_df[target_col]

# Identify numeric and categorical columns
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines for both numeric and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# Combine transformers into a preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Create a pipeline that includes preprocessing and the model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear'))
])

# Train the model
model_pipeline.fit(train_inputs, train_targets)

# Helper function to predict, compute accuracy & plot confusion matrix
def predict_and_plot(inputs, targets, name=''):
    preds = model_pipeline.predict(inputs)
    f1 = f1_score(targets, preds, pos_label='Yes')
    print("F1 score: {:.2f}%".format(f1))
    cf = confusion_matrix(targets, preds, normalize='true')
    plt.figure()
    sns.heatmap(cf, annot=True, cmap='Blues')
    plt.xlabel('Prediction')
    plt.ylabel('Target')
    plt.title('{} Confusion Matrix'.format(name))
    plt.show()
    return preds

# Evaluate on validation and test set
val_preds = predict_and_plot(val_inputs, val_targets, 'Validation')
test_preds = predict_and_plot(test_inputs, test_targets, 'Test')

# Save the trained model
joblib.dump(model_pipeline, PROCESSED_DATA_DIR / "aussie_rain_pipeline.joblib")

# Load the trained model
model_pipeline_loaded = joblib.load(PROCESSED_DATA_DIR / "aussie_rain_pipeline.joblib")
